In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GamingStocksInspection") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark session started.")


In [ ]:
import os

folder = "/Users/venubandi/Desktop/DATA_603_/expanded_csvs"

# List only CSVs
files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])

print("Number of files:", len(files))
for f in files:
    print(f)


In [ ]:
from pyspark.sql import DataFrame

for f in files:
    path = os.path.join(folder, f)
    print("\n===================================")
    print("FILE:", f)
    print("===================================")

    try:
        df_temp = spark.read.csv(
            path,
            header=True,
            inferSchema=True
        )
        df_temp.show(5)
        print("Columns:", df_temp.columns)

    except Exception as e:
        print("ERROR READING FILE:", f)
        print(e)


In [ ]:
from pyspark.sql.functions import col, to_date
from functools import reduce
from pyspark.sql import DataFrame
import os

folder = "/Users/venubandi/Desktop/DATA_603_/expanded_csvs"
files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])

dfs = []

for f in files:
    path = os.path.join(folder, f)
    print("Cleaning:", f)

    # Load CSV
    df_temp = spark.read.csv(path, header=True, inferSchema=True)

    # Fix wrong column name
    if "Price" in df_temp.columns:
        df_temp = df_temp.withColumnRenamed("Price", "Date")

    # Remove two corrupted header rows
    df_temp = df_temp.filter(~col("Date").isin("Ticker", "Date"))

    # Convert date column
    df_temp = df_temp.withColumn("Date", to_date("Date"))

    # Append to list
    dfs.append(df_temp)

# Combine all cleaned dataframes
df_all = reduce(DataFrame.unionByName, dfs)

print("Combined dataset:")
df_all.show(10)
df_all.printSchema()


In [ ]:
df_all.count()


In [ ]:
from pyspark.sql import DataFrame
import os

folder = "/Users/venubandi/Desktop/DATA_603_/expanded_csvs"
files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])

total_before = 0

for f in files:
    path = os.path.join(folder, f)

    df_temp = spark.read.csv(path, header=True, inferSchema=True)
    total_before += df_temp.count()

print("TOTAL ROWS BEFORE CLEANING:", total_before)


In [ ]:
from pyspark.sql.functions import col
import os

folder = "/Users/venubandi/Desktop/DATA_603_/expanded_csvs"
files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])

total_null = 0
total_non_null = 0
total_rows = 0

for f in files:
    path = os.path.join(folder, f)
    df_temp = spark.read.csv(path, header=True, inferSchema=True) \
                         .withColumnRenamed("Price", "Date")

    null_c = df_temp.filter(col("Date").isNull()).count()
    non_null_c = df_temp.filter(col("Date").isNotNull()).count()
    total_c = df_temp.count()

    total_null += null_c
    total_non_null += non_null_c
    total_rows += total_c

    print(f"\n{f}")
    print("  NULL Date     :", null_c)
    print("  NON-NULL Date :", non_null_c)
    print("  TOTAL         :", total_c)

print("\n============= COMBINED TOTAL =============")
print("TOTAL NULL dates    :", total_null)
print("TOTAL NON-NULL dates:", total_non_null)
print("TOTAL ROWS          :", total_rows)


In [ ]:
df_test = spark.read.csv(
    "/Users/venubandi/Desktop/DATA_603_/expanded_csvs/AMD_stock_data_expanded.csv",
    header=True,
    inferSchema=True
).withColumnRenamed("Price", "Date")

df_test.show(10)
